
# Azolla Biofilter — Additional Computational Experiments

This Colab notebook adds three **computational experiments** to strengthen the manuscript:

1. **Sobol global sensitivity analysis**
2. **Velocity-dependent biological-effectiveness scenarios**
3. **CFD scalar time-step independence**

These are numerical/modeling studies, not experimental validation of Azolla gas-phase CO₂ capture.


In [ ]:

# Install/import
import sys, subprocess, pkgutil, os
if pkgutil.find_loader("SALib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "SALib"])

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from SALib.sample import saltelli
from SALib.analyze import sobol

OUT = Path("Azolla_Computational_Experiments_Outputs")
OUT.mkdir(exist_ok=True)

V=36.0
ACH0=2.0
Q0=ACH0*V/3600
C0=600.0
Cout=400.0
A0=0.18
K0=1e-9
CF0=0.30
Lpanel=0.05
mu=1.81e-5
rho=1.225
eta_f0=0.60
sigma0=0.30
t30=1800.0
print("Q =",Q0,"m3/s")



## 1. Sobol global sensitivity analysis

Inputs:
- σ: 0.15–0.45
- ACH: 1.5–2.5 h⁻¹
- K: 0.8–1.2×10⁻⁹ m²
- C_F: 0.05–0.50
- A: 0.05–0.30 m²
- fan efficiency: 0.54–0.66

Outputs:
- 30-min excess-CO₂ removal
- pressure drop
- panel-related fan power


In [ ]:

problem={
 'num_vars':6,
 'names':['sigma','ACH','K','CF','A','eta_f'],
 'bounds':[[0.15,0.45],[1.5,2.5],[0.8e-9,1.2e-9],[0.05,0.50],[0.05,0.30],[0.54,0.66]]
}
X=saltelli.sample(problem,1024,calc_second_order=False)

def outputs(r):
    sigma,ACH,K,CF,A,eta_f=r
    Q=ACH*V/3600
    lam=ACH/3600
    kbio=lam*sigma
    eta30=1-np.exp(-(lam+kbio)*t30)
    uf=Q/A
    dp=((mu/K)*uf+0.5*rho*CF*uf**2)*Lpanel
    pf=Q*dp/eta_f
    return eta30,dp,pf

Y=np.array([outputs(r) for r in X])
names=['eta30','pressure_drop','fan_power']
rows=[]
for j,name in enumerate(names):
    Si=sobol.analyze(problem,Y[:,j],calc_second_order=False,print_to_console=False)
    for p,s1,st in zip(problem['names'],Si['S1'],Si['ST']):
        rows.append([name,p,s1,st])
sob=pd.DataFrame(rows,columns=['output','parameter','S1','ST'])
sob.to_csv(OUT/'Sobol_indices.csv',index=False)
display(sob)

pivot=sob.pivot(index='parameter',columns='output',values='ST')
pivot.plot(kind='bar',figsize=(9,5))
plt.ylabel('Total-order Sobol index')
plt.title('Global sensitivity of model outputs')
plt.tight_layout()
plt.savefig(OUT/'Fig_Sobol_Global_Sensitivity.png',dpi=300,bbox_inches='tight')
plt.show()



## 2. Velocity-dependent σ scenario analysis

The current constant-σ formulation forces panel-area cancellation. To test model-form robustness, use several **illustrative scenario laws** for σ(u_f). These are not calibrated Azolla kinetics.

The objective is to show whether the nominal degenerate optimization disappears once biological capture depends on face velocity.


In [ ]:

Agrid=np.linspace(0.05,0.30,160)
uref=Q0/A0

def sigma_const(u): return np.full_like(np.asarray(u),0.30,dtype=float)
def sigma_mild(u):
    u=np.asarray(u); return 0.30/(1+0.6*np.maximum(u/uref-1,0))
def sigma_strong(u):
    u=np.asarray(u); return 0.30/(1+1.6*np.maximum(u/uref-1,0))
sigma_max=0.42
beta=-uref*np.log(1-0.30/sigma_max)
def sigma_sat(u):
    u=np.asarray(u); return sigma_max*(1-np.exp(-beta/np.maximum(u,1e-9)))

laws={
 'Constant sigma':sigma_const,
 'Mild velocity penalty':sigma_mild,
 'Strong velocity penalty':sigma_strong,
 'Saturating residence-time law':sigma_sat
}

rows=[]
for label,fn in laws.items():
    for A in Agrid:
        uf=Q0/A
        sig=float(fn(np.array([uf]))[0])
        lam=ACH0/3600
        kbio=(A/V)*uf*sig
        eta=1-np.exp(-(lam+kbio)*t30)
        dp=((mu/K0)*uf+0.5*rho*CF0*uf**2)*Lpanel
        rows.append([label,A,uf,sig,eta,dp])

df=pd.DataFrame(rows,columns=['scenario','A_m2','face_velocity_mps','sigma','eta30','pressure_drop_Pa'])
df.to_csv(OUT/'Velocity_Dependent_Sigma_Scenarios.csv',index=False)

plt.figure(figsize=(8,5))
for label,g in df.groupby('scenario'):
    plt.plot(g.A_m2,100*g.eta30,label=label)
plt.xlabel('Panel area, A (m²)')
plt.ylabel('30-min excess-CO₂ removal (%)')
plt.title('Area-performance response under sigma(u) scenarios')
plt.legend()
plt.tight_layout()
plt.savefig(OUT/'Fig_SigmaVelocity_AreaVsRemoval.png',dpi=300,bbox_inches='tight')
plt.show()

plt.figure(figsize=(8,5))
for label,g in df.groupby('scenario'):
    plt.plot(g.pressure_drop_Pa,100*g.eta30,label=label)
plt.xlabel('Pressure drop (Pa)')
plt.ylabel('30-min excess-CO₂ removal (%)')
plt.title('Performance-hydraulic trade-off under sigma(u) scenarios')
plt.legend()
plt.tight_layout()
plt.savefig(OUT/'Fig_SigmaVelocity_Tradeoff.png',dpi=300,bbox_inches='tight')
plt.show()



## 3. Time-step independence study for scalar transport

Run this section after the corrected CFD notebook, or replace the placeholders below with the same fixed airflow field used there.

Recommended time steps: **2.5, 5, 10, and 20 s**.

For each time step, record:
- 30-min room-mean CO₂
- occupant-zone CO₂
- spatial standard deviation
- difference relative to the smallest time step

A change below about 1% across the tested stable range is a useful numerical robustness check.


In [ ]:

dt_results = pd.DataFrame({
    'dt_s':[2.5,5.0,10.0,20.0],
    'room_mean_30min_ppm':[np.nan]*4,
    'occupant_zone_30min_ppm':[np.nan]*4,
    'spatial_std_30min_ppm':[np.nan]*4
})
display(dt_results)
dt_results.to_csv(OUT/'CFD_TimeStep_Independence.csv',index=False)



## 4. Manuscript interpretation

Use the following framing after running the notebook:

- Sobol results are **global sensitivity diagnostics**, not validation.
- Velocity-dependent σ laws are **scenario experiments**, not measured biological kinetics.
- If only constant σ produces a flat area-removal response, state that area cancellation is a **model-form diagnostic** rather than a physical discovery.
- Time-step independence complements the existing grid-independence and diffusivity-sensitivity checks.
- None of these analyses replaces experimental calibration of σ, K, or C_F.


In [ ]:

import shutil, os
if os.path.exists("Azolla_Computational_Experiments_Outputs.zip"):
    os.remove("Azolla_Computational_Experiments_Outputs.zip")
shutil.make_archive("Azolla_Computational_Experiments_Outputs","zip",root_dir=OUT)
print("Created Azolla_Computational_Experiments_Outputs.zip")
